# AIGC 图像频谱指纹挖掘与伪造检测

本 Notebook 是期末提交材料的可读复现入口。它不重新跑大规模训练，只读取 `report/tables` 和 `report/figures` 中已经生成的结果资产，展示数据协议、最终 clean 结果、鲁棒性诊断、可解释性图表和复现实验命令。

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
REPORT = ROOT / 'report'
TABLES = REPORT / 'tables'
FIGURES = REPORT / 'figures'

def show_csv(name, n=20):
    path = TABLES / name
    if not path.exists():
        print(f'Missing: {path}')
        return None
    df = pd.read_csv(path)
    display(df.head(n))
    return df

def show_fig(name):
    path = FIGURES / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'Missing: {path}')

## 1. 数据协议

实验使用 GenImage 中 ADM、BigGAN、VQDM 和 GLIDE 四个生成源。每个生成源包含 `train/ai`、`train/nature`、`val/ai`、`val/nature`。`train` 用于训练，`val` 作为测试集。二分类任务合并 AI 与 nature；生成源归因任务只使用 AI 图像。

In [ ]:
dataset_counts = show_csv('dataset_counts.csv')

## 2. 最终 clean 主结果

最终主结果采用 `outputs_v2_full_best`：`fusion_freq + flat LightGBM + wide profile + full train + no augmentation`。该路线在 clean validation 上达到二分类 macro-F1 0.9913、归因 macro-F1 0.9991、双任务平均 macro-F1 0.9952。

In [ ]:
v2 = show_csv('optimization_v2_summary.csv', n=30)
if v2 is not None:
    cols = [
        'run', 'feature_profile', 'model_architecture', 'sample_fraction',
        'binary_ai_vs_nature_macro_f1', 'ai_subsource_attribution_macro_f1',
        'combined_macro_f1', 'selected'
    ]
    display(v2[cols].sort_values('combined_macro_f1', ascending=False).head(10))

In [ ]:
show_fig('optimization_v2_macro_f1.png')
show_fig('scaleup_macro_f1.png')

## 3. 鲁棒性诊断

鲁棒性实验不重新训练模型，而是加载保存好的 best model，对 20% validation 子集施加 JPEG、resize 和 Gaussian noise。结论是：`fusion_freq` clean 很强，但对后处理扰动敏感；旧灰度 baseline clean 较低，却在部分 JPEG/noise 场景更稳。

In [ ]:
robust_cmp = show_csv('robustness_comparison.csv', n=25)
if robust_cmp is not None:
    summary = robust_cmp.pivot_table(
        index=['task', 'attack', 'level'],
        columns='run',
        values='macro_f1',
        aggfunc='first'
    ).reset_index()
    display(summary.head(30))

In [ ]:
show_fig('robustness_comparison_binary_ai_vs_nature.png')
show_fig('robustness_comparison_ai_subsource_attribution.png')

## 4. 混淆矩阵与特征解释

混淆矩阵用于检查错误集中在哪些类别；feature importance 用于定位模型依赖的频谱统计。最终模型的归因错误很少，但鲁棒性实验显示这些频谱统计在 degraded 图像中会发生明显分布偏移。

In [ ]:
show_fig('confusion_binary_ai_vs_nature_lightgbm.png')
show_fig('confusion_ai_subsource_attribution_lightgbm.png')
show_fig('feature_importance_binary_ai_vs_nature_top20.png')

In [ ]:
show_csv('top_features_binary_ai_vs_nature.csv', n=20)
show_csv('top_features_ai_subsource_attribution.csv', n=20)

## 5. 复现命令

以下命令是最终报告使用的核心流程。数据集不进入 Git 仓库，默认路径为 `C:/Users/99303/git/GenImage_data`。

In [ ]:
print(r'''
$DATA = 'C:/Users/99303/git/GenImage_data'

# 重新生成报告资产
.\.venv\Scripts\python.exe scripts\build_report_assets.py `
  --dataset-root $DATA `
  --report-dir report `
  --primary-output outputs_v2_full_best `
  --robustness-output outputs_v2_full_best_robust_20pct `
  --robustness-compare-outputs outputs_v2_full_best_robust_20pct outputs_4gen_full_best_robust_20pct

# 只评估保存模型的鲁棒性，不重新训练
.\.venv\Scripts\python.exe scripts\evaluate_best_robustness.py `
  --dataset-root $DATA `
  --model-output outputs_v2_full_best `
  --out-dir outputs_v2_full_best_robust_20pct `
  --sample-fraction 0.20 `
  --sample-seed 42 `
  --tasks both `
  --num-workers 16 `
  --feature-chunksize 64 `
  --robust-cache-dir robustness_cache_fusion
''')

## 6. 局限性

本项目不声明完整 GenImage benchmark，只使用四个生成源。模型是浅层频域检测器，不与深度检测器竞争 SOTA。鲁棒性结果说明当前最佳 clean 模型对后处理扰动敏感，后续如果要面向真实平台部署，应单独研究鲁棒训练、测试时增强或更稳健的低/中频特征融合。